# Flood impact on ER travel time - Benton, Washington and Madison Counties, AR

This notebook runs the full pipeline from the `claude/flood-impact-er-travel-time-s5goww` branch of
[jccrews256/Mapping-fun-](https://github.com/jccrews256/Mapping-fun-) in Colab, where FEMA's servers are reachable.

What it does, in order:

1. Clone the branch and install the geospatial stack.
2. (Optional) Mount Google Drive so the 3 GB of downloads and the outputs survive the session.
3. Confirm FEMA's NFHL service answers, and download the flood zones, BFE lines and cross-sections for the study box.
4. Run the depth-based analysis (Overture roads + ERs, USGS 3DEP elevation, FEMA water surface).
5. Run three sensitivity scenarios (12 in threshold, planimetric upper bound, bridges never closed).
6. Show the county summaries, the method flags and the maps.
7. Save the outputs to Drive or download them, and optionally commit them back to the branch.

**Runtime:** a normal CPU runtime is enough. Expect 15-25 minutes for the first run (about 3 GB of downloads:
five USGS DEM tiles plus the NFHL), then 5-8 minutes per extra scenario because everything is cached in `data/`.
If Colab reports it ran out of RAM, switch to a High-RAM runtime (Runtime -> Change runtime type).

## 1. Get the code and install dependencies

In [ ]:
%cd /content
!rm -rf nwa-flood
!git clone -q -b claude/flood-impact-er-travel-time-s5goww https://github.com/jccrews256/Mapping-fun-.git nwa-flood
%cd /content/nwa-flood
!pip install -q -r requirements.txt
import geopandas, rasterio, networkx, pyarrow
print("geopandas", geopandas.__version__, "| rasterio", rasterio.__version__, "| networkx", networkx.__version__, "| pyarrow", pyarrow.__version__)

## 2. (Optional) Keep downloads and outputs on Google Drive

Set `USE_DRIVE = True` to store `data/` (Overture pulls, NFHL, DEM tiles) and `output/` in
`My Drive/nwa-flood/`. A second session then skips every download.

In [ ]:
USE_DRIVE = False   # <- set True to persist data/ and output/ on Google Drive

import os, pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    root = pathlib.Path("/content/drive/MyDrive/nwa-flood")
    for sub in ["data", "output"]:
        (root / sub).mkdir(parents=True, exist_ok=True)
        if os.path.islink(sub) or os.path.exists(sub):
            os.system(f"rm -rf {sub}")
        os.symlink(root / sub, sub)
    print("data/ and output/ now live in", root)
else:
    print("using the Colab session disk (lost when the runtime recycles)")

## 3. Check that FEMA's NFHL service answers

The script discovers the layer ids by name. This cell shows the names it will match against
(`Flood Hazard Zones`, `Base Flood Elevations`, `Cross-Sections`) and confirms the query endpoint works.
If this cell fails, jump to **Option B** below and feed the script a downloaded NFHL file instead.

In [ ]:
import requests, pandas as pd
NFHL = "https://hazards.fema.gov/arcgis/rest/services/public/NFHL/MapServer"
svc = requests.get(NFHL, params={"f": "json"}, timeout=120).json()
layers = pd.DataFrame(svc["layers"])[["id", "name"]]
print(layers.to_string(index=False))
zones = layers[layers["name"].str.contains("Flood Hazard Zones", case=False)]["id"].iloc[0]
probe = requests.get(f"{NFHL}/{zones}/query", params={"where": "SFHA_TF = 'T'", "geometry": "-94.3,36.0,-94.1,36.2",
                     "geometryType": "esriGeometryEnvelope", "inSR": 4326, "returnCountOnly": "true", "f": "json"}, timeout=120).json()
print("\nSFHA polygons in a test box around Fayetteville:", probe)

## 4. Run the analysis (Option A: NFHL straight from FEMA)

Default settings: 1 km grid, 10 m DEM samples, a road closes at 0.15 m (6 in) of water on the pavement,
bridges judged by their abutments. Output goes to `output/depth_0.15/`.

The log prints each stage with a running clock. The NFHL and DEM downloads happen inside the script and are cached.

In [ ]:
!python flood_er_travel_time.py --out output/depth_0.15

### Option B: use an NFHL file you downloaded

Only if Option A could not reach FEMA. Get the Arkansas state NFHL from https://msc.fema.gov
(Search All Products -> Effective Products -> NFHL Data-State -> `NFHL_05_<date>.zip`, about 1 GB),
put it on Drive or upload it here, and point the script at it. The script reads `S_FLD_HAZ_AR`, `S_BFE`
and `S_XS` from the zip.

In [ ]:
NFHL_FILE = ""   # e.g. "/content/drive/MyDrive/NFHL_05_20250901.zip"; leave empty to upload interactively

if not NFHL_FILE:
    from google.colab import files
    up = files.upload()
    NFHL_FILE = next(iter(up)) if up else ""
if NFHL_FILE:
    !python flood_er_travel_time.py --nfhl "{NFHL_FILE}" --out output/depth_0.15
else:
    print("no file given - skipping Option B")

## 5. Sensitivity scenarios

Everything is cached now, so each run is the routing and depth stages only.

| scenario | flag | question it answers |
|---|---|---|
| `depth_0.30` | `--passable-depth 0.30` | how much changes if a foot of water is the cut-off |
| `rule_2d` | `--closure-rule 2d` | the planimetric upper bound (every edge touching the floodplain closed) |
| `bridges_open` | `--bridges-passable` | how much of the impact is bridge decks |

In [ ]:
NFHL_ARG = f'--nfhl "{NFHL_FILE}"' if globals().get("NFHL_FILE") else ""
!python flood_er_travel_time.py {NFHL_ARG} --passable-depth 0.30 --out output/depth_0.30
!python flood_er_travel_time.py {NFHL_ARG} --closure-rule 2d   --out output/rule_2d
!python flood_er_travel_time.py {NFHL_ARG} --bridges-passable  --out output/bridges_open

## 6. Results

In [ ]:
import pandas as pd, glob, os
pd.set_option("display.width", 220)
rows = []
for d in sorted(glob.glob("output/*/summary_by_county.csv")):
    s = pd.read_csv(d).assign(scenario=os.path.basename(os.path.dirname(d)))
    rows.append(s)
summary = pd.concat(rows).set_index(["scenario", "county"])
display(summary)
summary.to_csv("output/summary_all_scenarios.csv")

In [ ]:
flags = pd.read_csv("output/depth_0.15/method_flags.csv")
pd.set_option("display.max_colwidth", 200)
display(flags)

In [ ]:
from IPython.display import Image, display
for p in ["output/depth_0.15/map_baseline_minutes.png", "output/depth_0.15/map_flood_increase.png",
          "output/rule_2d/map_flood_increase.png"]:
    print(p); display(Image(p, width=900))

In [ ]:
# which ERs serve how many grid points, before and after
pd.read_csv("output/depth_0.15/nearest_er_counts.csv", index_col=0).sort_values("baseline", ascending=False)

In [ ]:
# the floodplain edges with their water-surface source, road elevation, depth and closure flags
import geopandas as gpd
fe = gpd.read_file("output/depth_0.15/floodplain_road_edges.gpkg")
print(len(fe), "edges touch the floodplain")
display((fe.groupby(["wse_source", "closed_depth"])["length_m"].sum() / 1609.344).round(0).unstack(fill_value=0).rename(columns={False: "open (mi)", True: "closed (mi)"}))
fe["depth_max_m"].describe().round(2)

## 7. Review the ER roster

`ER_TABLE` in the script is a judgement call. Check the addresses and the `include` column; edit the table
in `flood_er_travel_time.py` and re-run section 4 if something is wrong.

In [ ]:
pd.read_csv("output/depth_0.15/er_locations.csv")[["er_name", "county", "include", "note", "address", "snap_m"]]

## 8. Save the outputs

In [ ]:
!rm -f nwa_flood_outputs.zip
!zip -qr nwa_flood_outputs.zip output -x "*.parquet"
from google.colab import files
files.download("nwa_flood_outputs.zip")

### (Optional) Commit the results back to the branch

Needs a GitHub personal access token with `repo` scope (entered with a hidden prompt, never stored).
Only the CSVs, PNGs and flags are committed; the GeoPackages and samples are large and stay local.

In [ ]:
COMMIT_RESULTS = False   # <- set True to push

if COMMIT_RESULTS:
    from getpass import getpass
    token = getpass("GitHub token: ")
    !git config user.name "jccrews256"
    !git config user.email "jccrews@ncsu.edu"
    !git add -f output/*/summary_by_county.csv output/*/nearest_er_counts.csv output/*/method_flags.csv output/*/er_locations.csv output/*/grid_flood_impact.csv output/*/*.png output/summary_all_scenarios.csv
    !git commit -q -m "Add flood-impact results from Colab run (FEMA NFHL + 3DEP)"
    !git push https://{token}@github.com/jccrews256/Mapping-fun-.git claude/flood-impact-er-travel-time-s5goww
    del token